# Demo: Sistema RAG para EcoMarket con Llama 3.1 8B
Este notebook muestra el flujo completo de un sistema RAG local usando los datos reales del taller.

In [1]:
# Si aún no tienes las dependencias, descomenta la siguiente línea:
# !pip install langchain-community chromadb ollama

## 1. Importar librerías y verificar Ollama

In [2]:
import subprocess
import sys
from pathlib import Path
import csv
import json

try:
    subprocess.run(["ollama", "--version"], capture_output=True, check=True)
    print("✅ Ollama está disponible")
except Exception as exc:
    print("❌ Ollama no está disponible:", exc)
    print("Instala Ollama y asegúrate de que esté en el PATH.")
    sys.exit(1)

from langchain.chains import RetrievalQA
from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.llms import Ollama
from langchain.vectorstores import Chroma

✅ Ollama está disponible


## 2. Cargar los datos del taller

In [3]:
BASE_DIR = Path().resolve()
DATA_DIR = BASE_DIR / "data"
print("Archivos disponibles:")
for file in sorted(DATA_DIR.iterdir()):
    print("-", file.name)

with open(DATA_DIR / "politicas_ecomarket.txt", encoding="utf-8") as f:
    politicas = f.read().strip()
print("\nPolíticas cargadas (primeros 300 caracteres):")
print(politicas[:300], "...")

Archivos disponibles:
- faqs.json
- inventario_productos.csv
- politicas_ecomarket.txt

Políticas cargadas (primeros 300 caracteres):
POLÍTICA DE DEVOLUCIONES DE ECOMARKET:

1. PLAZO DE DEVOLUCIÓN:
Tienes 30 días desde la fecha de compra para devolver un producto. Este es el período estándar de devolución en EcoMarket.

2. CONDICIONES PARA DEVOLVER UN PRODUCTO:
- El producto debe estar sin usar.
- El artículo debe estar en su emba ...


In [4]:
with open(DATA_DIR / "faqs.json", encoding="utf-8") as f:
    faqs = json.load(f)
print("\nFAQs cargadas:")
for faq in faqs:
    print(f"- {faq['pregunta']} -> {faq['respuesta']}")


FAQs cargadas:
- ¿Cuánto tiempo tengo para devolver un producto? -> Tienes 30 días desde la fecha de compra para devolver un producto. El artículo debe estar sin usar, en su embalaje original y en las mismas condiciones en que lo recibiste.
- ¿Aceptan devolución de alimentos? -> No. Los productos perecederos, como alimentos, bebidas y flores, NO son elegibles para devolución. Esto se debe a que no podemos garantizar las condiciones de almacenamiento y calidad del producto.
- ¿Cuánto tardan los envíos? -> Los envíos a Cali tardan 2 días hábiles. A otras ciudades de Colombia, entre 3 y 5 días hábiles.
- ¿Qué métodos de pago aceptan? -> Aceptamos tarjetas de crédito (Visa, Mastercard), Mercado Pago, y transferencia bancaria.
- ¿Puedo cambiar un producto? -> Sí, tienes 15 días desde la entrega. El producto debe estar sin usar y en su empaque original. No aplica para productos de higiene personal o perecederos abiertos.
- ¿Qué productos ecológicos tienen en inventario? -> EcoMarket ofrece 

In [5]:
productos = []
with open(DATA_DIR / "inventario_productos.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        productos.append(row)
print(f"\nProductos cargados: {len(productos)}")
print(productos[0])


Productos cargados: 10
{'id': 'PROD-001', 'nombre': 'Termo de acero inoxidable 500ml', 'categoria': 'Accesorios', 'stock': '45', 'precio': '35000', 'descripcion': 'Termo reutilizable para bebidas calientes y frías. Material ecológico', 'atributo_ecologico': 'Acero reciclable'}


## 3. Preparar documentos y crear la base vectorial

In [6]:
def build_documents():
    documents = []

    documents.append(Document(
        page_content=politicas,
        metadata={"source": "politicas_ecomarket.txt", "type": "politica"}
    ))

    for item in faqs:
        documents.append(Document(
            page_content=f"Pregunta: {item['pregunta']}\nRespuesta: {item['respuesta']}",
            metadata={"source": "faqs.json", "type": "faq"}
        ))

    for row in productos:
        documents.append(Document(
            page_content=(
                f"ID: {row['id']}\n"
                f"Nombre: {row['nombre']}\n"
                f"Categoría: {row['categoria']}\n"
                f"Precio: {row['precio']}\n"
                f"Stock: {row['stock']}\n"
                f"Atributo ecológico: {row['atributo_ecologico']}\n"
                f"Descripción: {row['descripcion']}"
            ),
            metadata={"source": "inventario_productos.csv", "type": "producto", "id": row['id']}
        ))

    return documents


docs = build_documents()
print(f"Documentos preparados: {len(docs)}")

Documentos preparados: 17


In [7]:
PERSIST_DIR = BASE_DIR / "chroma_db"
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/distiluse-base-multilingual-cased-v2")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=256,
    chunk_overlap=64,
    separators=["\n\n", "\n", ".", " ", ""]
)
chunks = splitter.split_documents(docs)
print(f"Chunks creados: {len(chunks)}")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(PERSIST_DIR),
)
vectorstore.persist()
print("Vectorstore persistida en disco en:", PERSIST_DIR)

c:\Users\14624165\GitHub\IAGenerativa\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8148.39it/s]


Chunks creados: 30
Vectorstore persistida en disco en: C:\Users\14624165\GitHub\IAGenerativa\Proyecto\Taller_2\chroma_db


## 4. Construir la cadena RAG y hacer consultas

In [8]:
llm = Ollama(model="llama3.1:8b", temperature=0.1)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

prompt = PromptTemplate(
    template="""
Usa SOLO el siguiente contexto para responder.
Si no hay suficiente información, responde:
"No tengo información suficiente. Un asesor humano te contactará."

Contexto:
{context}

Pregunta: {question}
Respuesta:
""",
    input_variables=["context", "question"],
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    verbose=False,
)
print("Cadena RAG lista.")

Cadena RAG lista.


In [9]:
for pregunta in [
    "¿Cuánto tiempo tengo para devolver un producto?",
    "¿Aceptan devolución de alimentos?",
    "¿Qué métodos de pago aceptan?",
    "¿Qué productos ecológicos tienen en inventario?"
]:
    respuesta = qa_chain.run(pregunta)
    print(f"\nPregunta: {pregunta}\nRespuesta: {respuesta}")

c:\Users\14624165\GitHub\IAGenerativa\.venv312\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(



Pregunta: ¿Cuánto tiempo tengo para devolver un producto?
Respuesta: Tienes 30 días desde la fecha de compra para devolver un producto. El artículo debe estar sin usar, en su embalaje original y en las mismas condiciones en que lo recibiste.

Pregunta: ¿Aceptan devolución de alimentos?
Respuesta: No. Los productos perecederos, como alimentos, bebidas y flores, NO son elegibles para devolución. Esto se debe a que no podemos garantizar las condiciones de almacenamiento y calidad del producto.

Pregunta: ¿Qué métodos de pago aceptan?
Respuesta: Aceptamos tarjetas de crédito (Visa, Mastercard), Mercado Pago, y transferencia bancaria.

Pregunta: ¿Qué productos ecológicos tienen en inventario?
Respuesta: EcoMarket ofrece una amplia variedad de productos ecológicos: Termo de acero reciclable, Chaqueta de algodón orgánico, Cepillo de bambú biodegradable, Shampoo sólido zero waste, Mochila de cáñamo sostenible, Jabón líquido circular, Blusa de algodón ecológica.


## 5. Uso del script

Puedes ejecutar este demo con el script de Python:

```bash
cd Proyecto/Taller_2/scripts
python rag_llama3.1.py --rebuild
```

Y luego lanzar una consulta interactiva o usar `--query` para una sola pregunta.